# **Data Load**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# google drive에 있는 dataset.zip 압축 풀어서 /content/ 아래 저장
!unzip -q "/content/drive/MyDrive/Autonomous_dataset.zip" -d "/content/dataset/"

# **전처리**

In [ ]:
import pandas as pd
import numpy as np
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

In [ ]:
# csv파일에서 offset, 헤딩각도 및 tuple 형태의 데이터에서 x좌표를 추출하여 하나의 label list로 묶어주는 함수
# dataframe 형태로 인자(row_lables)를 받음
def tuple_to_string(row_labels):

    # csv의 1, 2번째 col의 label은 형태 그대로 저장
    result_list = [float(row_labels[1]), float(row_labels[2])]

    # (x, y) 형태로 되어 있는 열 3 ~ 11의 값을 float형태의 두 값으로 추출하여 x값만을 xs list에 수집
    xs=[]

    for i in range(3, 12):
        val = str(row_labels[i])

        # (, )를 제거하고 :를 기준으로 값 나누기
        parts = val.replace('(', '').replace(')','').split(':')

        # x값만을 xs list에 추가
        xs.append(float(parts[0]))

    base=xs[0]

    # x0를 기준으로 나머지 좌표의 상대좌표를 result_list에 저장
    for x in xs:
        result_list.append(x-base)

    return result_list

# **Dataset, DataLoader 정의**


In [ ]:
from torchvision import transforms

# 데이터 가공을 이한 전처리 파이프라인
transform=transforms.Compose([
    # 이미지를 텐서로 변환
    transforms.ToTensor(),
    # ImageNet 데이터셋의 평균과 표준편차를 동일하게 적용
    # ImageNet 데이터셋과 동일하게 전처리하는 이유는 전이학습에 사용한 ResNet50은 ImageNet 데이터셋을 통해 만들어진 모델이기 때문
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset 정의
from torch.utils.data import Dataset
from PIL import Image
import os


class MyDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform):
        self.df=pd.read_csv(csv_file, header=None)
        # 이미지 폴더 경로를 저장할 변수
        self.img_dir=img_dir
        self.transform=transform

        # csv의 모든 행에 대해 tuple_to_string 함수를 적용
        initial_labels=self.df.apply(tuple_to_string, axis=1).tolist()
        labels_numpy=np.array(initial_labels, dtype=np.float32)

        # 예측해야 할 feature들의 단위와 범위가 다르기 때문에 labels 정규화를 진행
        # 정규화를 하지 않을 경우 수치가 큰 feature의 오차에만 민감하게 반응하고 작은 값은 무시하게됨
        # Z-score 정규화
        self.label_mean=labels_numpy.mean(axis=0)
        self.label_std=labels_numpy.std(axis=0)

        # 오류를 막기 위해 표준편차가 0 인 값이 있다면 아주 작은 값으로 수정
        self.label_std[self.label_std==0]=1e-6

        # 데이터의 분포 모양을 유지하며 표준정규분포의 형식을 따르는 데이터로 정규화
        scaled_labels=(labels_numpy-self.label_mean)/self.label_std

        # tensor 형태로 변환
        self.labels=torch.tensor(scaled_labels, dtype=torch.float32)

        # csv파일의 첫번째 col을 통해 사진명 저장
        self.img_names=self.df.iloc[:, 0].values

        # 이후 메모리 효율을 위한 변수 삭제
        del self.df
        del initial_labels
        del labels_numpy

        # Garbage Collector
        # 현제 메모리에 쌓인 쓰레기값들을 즉시 처리하는 코드
        import gc
        gc.collect()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        # 인덱스에 맞는 이미지의 경로 저장
        img_path=os.path.join(self.img_dir, self.img_names[idx])

        # ImageNet 기반의 사전학습 모델은 반드시 3채널 이미지를 입력으로 받아야 함
        image=Image.open(img_path).convert('RGB')

        # 이미지 전처리
        image=self.transform(image)

        # 레이블 가져오기
        label=self.labels[idx]

        return image, label

    # 정규화되어 있는 예측값을 원래 값으로 돌려주는 함수
    def inverse_scale(self, scaled_val):
        # 값이 계산되려면 CPU or GPU 같은 장소에 있어야 함

        # 데이터가 텐서형태라면, 데이터가 올라가있는 위치인 scaled_val.device로 보냄
        if torch.is_tensor(scaled_val):
            l_mean=torch.tensor(self.label_mean).to(scaled_val.device)
            l_std=torch.tensor(self.label_std).to(scaled_val.device)
        else:
            # float, int 형태는 무조건 CPU에 있으므로 그대로 변수 정의
            l_mean, l_std = self.label_mean, self.label_std

        # Z-score로 정규화된 값을 다시 되돌려 반환
        return scaled_val*l_std+l_mean

In [ ]:
csv_path="/content/dataset/dataset/labels.csv"
img_path="/content/dataset/dataset/Images"

# 전체 데이터셋 객체 생성
dataset=MyDataset(csv_path, img_path, transform=transform)

# 크기 계산
data_size=len(dataset)
train_size=int(data_size*0.8)
test_size=int(data_size*0.1)
val_size=data_size-train_size-test_size

# 데이터 분할
from torch.utils.data import random_split
train_data, val_data, test_data=random_split(dataset,
                                             [train_size, val_size, test_size])

# 각각의 DataLoader 생성
# batch_size는 32개, colab의 4개의 gpu코어를 효율적으로 사용하기 위해 num_workers=4
from torch.utils.data import DataLoader
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

# **CNN 모델 및 Wrapper 모델 정의**

In [ ]:
import torch.nn as nn
from torchvision import models

class CNN_Model(nn.Module):
    def __init__(self, num_outputs=11):
        super(CNN_Model, self).__init__()
        # ResNet-50 불러오기
        self.backbone=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # ResNet-50의 마지막층인 fc가 입력으로 받는 데이터의 개수를 저장
        in_features=self.backbone.fc.in_features

        # ResNet-50의 마지막 층 fc를 아무것도 하지 않는 층인 nn.Identity()로 교체
        # Identity는 입력받은 값을 아무런 계산 없이 그대로 다음으로 넘겨주는 통로 역할을 함
        self.backbone.fc=nn.Identity()


        # 중앙차선 이탈 값(offset)을 위한 분류기
        # output : offset 1개
        self.head_offset = nn.Sequential(
            # 들어온 데이터의 차원을 256개로 확장
            nn.Linear(in_features, 256),
            # Batch Normalization : 데이터의 분포를 배치 단위로 정규화하여 학습속도 및 안정성 향상
            # 공변량 변화 : 딥러닝 모델의 각 층은 이전 층에서 나온 결과물을 입력으로 받음, 학습을 진행하며 앞쪽 층의 가중치가 변하면서 뒤쪽 층으로 들어가는 데이터의 분포가 계속 바뀌는 현상
            # 배치 정규화를 하지 않을경우 매번 새로운 분포에 적응해야 함.
            # 배치 정규화는 데이터를 일정한 평균과 분산으로 고정시킴.
            nn.BatchNorm1d(256),
            # 활성화함수
            nn.ReLU(),
            # 특정 뉴런에만 의존하는 과적합을 방지
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

        # 헤딩각도 오차값을 위한 분류기
        self.head_heading = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

        # 전방 10개의 좌표를 위한 분류기
        self.head_waypoints = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            # 중간에 128개의 은닉층을 하나 더 추가시켜 정밀도 상승
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),

            # 최종 9개 출력
            nn.Linear(128, 9),
        )

    def forward(self, x):

        # ResNet50 모델을 통해 이미지의 핵심적인 특징을 뽑아냄
        features=self.backbone(x)

        # 입력된 특징을 바탕으로 Offset 하나를 계산
        out_offset = self.head_offset(features)       # (Batch, 1)

        # 입력된 특징을 바탕으로 헤딩각도 하나를 계산
        out_heading = self.head_heading(features)     # (Batch, 1)

        # 입력된 특징을 바탕으로 좌표값 9개를 계산
        out_waypoints = self.head_waypoints(features) # (Batch, 9)

        # 각각의 결과들을 하나의 리스트로 이어붙여 반환
        return torch.cat([out_offset, out_heading, out_waypoints], dim=1)

# **배포를 위해 물리값을 뱉어내는 Wrapper 모델 정의**

In [ ]:
# 모델의 output으로는 정규화된 값이 나옴, 자율주행에 필요한 실제 물리값을 뱉어내기 위한 장치
class DeploymentModel(nn.Module):
    def __init__(self, trained_model, label_mean, label_std):
        super(DeploymentModel, self).__init__()
        self.trained_model = trained_model

        # register_buffer: 학습시키는 파라미터는 아니지만, 모델과 함께 저장되어야 하는 변수를 저장할때 사용
        # mean, std 값을 저장
        self.register_buffer('mean', torch.tensor(label_mean, dtype=torch.float32))
        self.register_buffer('std', torch.tensor(label_std, dtype=torch.float32))

        # 고정된 y값 저장 (1~9)
        # 배포 시 모델과 함께 저장되도록 register_buffer를 사용
        y_values=torch.arange(1, 10, dtype=torch.float32)
        self.register_buffer('fixed_y', y_values)

    def forward(self, x):
        # 모델에서 나온 정규화된 값을 저장
        normalized_out = self.trained_model(x)

        # 정규화된 값을 물리값으로 역스케일링
        out_11=normalized_out*self.std+self.mean

        # 최종 출력을 위한 빈 텐서 생성
        batch_size=x.size(0)
        final_output=torch.zeros((batch_size, 20), device=x.device)

        # 0, 1열은 각각 offset, heading 값 저장
        final_output[:, 0:2]=out_11[:, 0:2]

        # 2, 4, 6, 8, 10, 12, 14, 16, 18 열에는 모델이 예측한 x값을 저장
        final_output[:, 2::2]=out_11[:, 2:]
        # 3, 5, 7, 9, 11, 13, 15, 17, 19 열에는 고정된 정수값 y(1, 2, 3, ... 9)을 배치사이즈에 맞게 확장시켜 저장
        final_output[:, 3::2]=self.fixed_y.expand(batch_size, -1)


        # 최종 결과물
        # [offset, heading, x1, y1, x2, y2, x3, y3, x4, y4, ... x9, y9]
        return final_output

# **학습 및 검증 루프**

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

# amp 사용시 float16을 사용하게 됨 -> float16은 아주 작은 값을 0으로 처리해버리는 단점 존재
# scale : 계산된 오차에 아주 큰 숫자를 곱합 (float16에서도 0.0이 되지 않고 역전파과정을 안전하게 수행)
# unscale : 역전파과정이 끝난 후, 계산된 가중치를 수정하기 전에 곱했던 큰 숫자로 다시 나누어줌
scaler=torch.amp.GradScaler('cuda')

def train_model(model, loader, criterion, optimizer, device):

    # 모델을 학습 상태로 설정
    model.train()
    # 이번 epoch의 오차를 저장할 변수
    running_loss=0

    # tqdm을 통해 학습 진행상황을 시각화
    loop=tqdm(loader, leave=False)

    for images, labels in loop:

        # 데이터를 GPU로 이동
        images, labels=images.to(device), labels.to(device)

        # amp : float32 대신 float16을 사용하여 GPU 메모리 사용량을 줄이고 연산속도 향상시키는 기술
        with torch.amp.autocast('cuda'):

            # 모델의 이미지를 넣어 예측값 생성
            outputs=model(images)
            # 예측값과 정답을 비교하여 오차를 계산
            loss=criterion(outputs, labels)


        # 이전 batch에서 계산된 기울기를 초기화
        optimizer.zero_grad()

        # 오차를 scale(키우기) 한뒤 역전파 수행
        scaler.scale(loss).backward()
        # scale 되었던 기울기를 다시 되돌림
        scaler.unscale_(optimizer)
        # float16을 사용하면 아주 작은 숫자는 0으로 인식되어 학습이 멈출 수도 있음
        # 오차에 큰 숫자를 곱해서 역전파를 마친 뒤, 가중치를 다시 원래의 크기로 unscale하는 것


        # Gradient clipping
        # 사전에 정해둔 임계값을 기준으로 기울기 폭주를 막기 위한 장치
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 가중치 업데이트
        scaler.step(optimizer)
        scaler.update()

        # 화면에 출력할 이번 epoch의 오차값을 누적
        running_loss+=loss.item()

        # tqdm 바에 나타낼 정보들
        loop.set_description(f'Training')
        loop.set_postfix(loss=loss.item())

    # 전체 데이터셋을 다 돌면, 평균 오차를 반환
    return running_loss / len(loader)


def validation_model(model, loader, criterion, device):

    # 모델을 검증 상태로 설정
    model.eval()
    # 오차를 저장할 변수
    val_loss=0

    with torch.no_grad():

        # tqdm을 통해 학습 진행상황을 시각화
        loop=tqdm(loader, leave=False, desc='Validation')

        for images, labels in loop:

            # 데이터를 GPU로 이동
            images, labels=images.to(device), labels.to(device)

            # 예측값 저장
            outputs=model(images)

            # 예측값과 정답간의 오차를 저장
            loss=criterion(outputs, labels)

            # 저장된 오차를 누적
            val_loss+=loss.item()

            # tqdm 바에 나타낼 정보
            loop.set_postfix(val_loss=loss.item())

    # 평균 오차를 반환
    return val_loss/len(loader)

# **학습 과정**

In [ ]:
import gc
import torch

# cuda가 있으면, GPU를 사용하고 없으면 CPU를 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CNN_Model() 객체 생성 후 앞서 선택한 장치(GPU)의 메모리 위로 올려놓음
model = CNN_Model().to(device)

# MAE를 오차함수로 사용
# offset이나 좌표같은 '거리'나 '물리적 수치'를 예측하는 회귀문제에서는
# 너무 큰 오차가 발생했을때, 기울기 폭주를 막기 위해 MAE를 많이 사용
criterion = nn.L1Loss().to(device)

epochs=30

# Phase 1 : 특징추출기(몸통)을 고정시킴
for param in model.backbone.parameters():
    # 이 부분의 기울기는 계산 하지 않겠다는 의미의 코드
    param.requires_grad=False

# ResNet50은 이미 이미지의 특징을 파악하는 능력을 갖고있는 모델
# 본인이 새로 정의한 3개의 heads는 아무런 능력이 없는 무지 상태
# 만약 처음부터 둘을 동시에 학습시키면 heads가 만들어내는 엉뚱한 오차에 의해 backbone의 세팅값이 함께 망가질 수도 있음
# Phase 1 에서는 몸통은 얼린 상태로 heads를 먼저 학습시킴

# heads들의 가중치들이 학습되도록 함
for param in model.head_offset.parameters():
    param.requires_grad = True
for param in model.head_heading.parameters():
    param.requires_grad = True
for param in model.head_waypoints.parameters():
    param.requires_grad = True


head_params = list(model.head_offset.parameters()) + \
            list(model.head_heading.parameters()) + \
            list(model.head_waypoints.parameters())

# 몸통의 learning_rate=0, heads들의 learning_rate=0.001로 설정
optimizer=optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 0.0},
    {'params': model.head_offset.parameters(), 'lr': 1e-3},
    {'params': model.head_heading.parameters(), 'lr': 1e-3},
    {'params': model.head_waypoints.parameters(), 'lr': 1e-3}
])

# 에포크마다 학습율을 서서히 줄여주는 도구를 우선 사용하지 않겠다는 뜻
# heads만 빠르게 학습하는 Phase 1에서는 보통 고정된 학습율을 사용
scheduler=None

# 지금까지 산출된 validation loss 중 가장 작은 값을 저장할 변수
# 처음에는 비교대상이 없으므로 무한대(inf)로 설정해둠
best_val_loss=float('inf')

for epoch in range(epochs):

    # Phase 2 : epoch 5부터 고정시켜두었던 몸통(backbone)을 학습되도록 함
    # 이제 heads들이 어느정도 학습되었으니, 전체 모델을 자율주행 데이터에 맞게 조율하는 미세조정(Fine-Tuning)을 시작
    if epoch==5:
        for param in model.parameters():
            param.requires_grad=True

        # 옵티마이저들의 learning_rate를 변경
        # 이미 충분히 똑똑한 backbone은 0.00001, heads는 0.0001
        optimizer.param_groups[0]['lr'] = 1e-5
        optimizer.param_groups[1]['lr'] = 1e-4
        optimizer.param_groups[2]['lr'] = 1e-4
        optimizer.param_groups[3]['lr'] = 1e-4

        # validation loss가 3번 연속으로 감소하지 않으면, 학습율을 1/2로 깎는 스케쥴러 도입
        scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

        # 불필요한 메모리를 정리하는 코드
        gc.collect()
        torch.cuda.empty_cache()

    # 매 epoch 마다 train, validation 진행
    train_loss=train_model(model, train_loader, criterion, optimizer, device)
    val_loss=validation_model(model, val_loader, criterion, device)


    print(f"Epoch [{epoch}/{epochs-1}] - Train loss: {train_loss:.5f}, Val loss: {val_loss:.5f}")

    # epoch 5부턴 validation loss를 스케쥴러에게 전달하여 학습율을 깎을지 말지 판단하게 함
    if epoch>=5:
        scheduler.step(val_loss)

    # validation loss가 이전 epoch들 중 가장 낮다면
    if val_loss<best_val_loss:
        best_val_loss=val_loss

        # 모델을 검증 상태로 설정
        model.eval()

        # CNN_Model을 Wrapper로 감싸기
        deploy_model = DeploymentModel(model, dataset.label_mean, dataset.label_std).to(device)
        deploy_model.eval()

        # 가상의 이미지 입력데이터 만들기 (batch 1, channel 3, 224x224)
        example_input=torch.rand(1, 3, 224, 224).to(device)

        # 역전파 과정을 위한 연산 과정 기록을 하지 않음
        with torch.no_grad():
            # trace 함수는 example_input을 모델에 한번 통과시켜보며 모델 내부에서 일어나는 모든 과정을 녹화함
            traced_model=torch.jit.trace(deploy_model, example_input)

        # 모델 저장
        traced_model.save("/content/drive/MyDrive/best_resnet_model.pt")
        print("모델 저장됨")

# **모델 예측값 검증(MAE)**

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm

def calculate_feature_mae(model_path, test_loader, device):
    # torch.jit.load를 통해 모델 불러와서 GPU에 올려놓기
    model = torch.jit.load(model_path).to(device)
    # 모델을 검증 상태로 설정
    model.eval()

    # 결과값을 기록할 0으로 채워진 (,11) 형태의 텐서를 만들어 GPU에 올림
    total_abs_error = torch.zeros(11).to(device)
    # MAE를 구하기 위해, 몇장의 이미지를 처리했는지 기록할 변수
    total_samples = 0

    # 기울기 계산을 배제하여 메모리를 아끼고 연산속도 향상
    with torch.no_grad():

        # test_loader로부터 Batch 단위로 이미지와 정답을 가져옴
        for images, labels_normalized in tqdm(test_loader, desc="Evaluating"):
            # 가져온 이미지와 정답을 GPU로 올림
            images = images.to(device)
            labels_normalized = labels_normalized.to(device)

            # 이번 Batch에 이미지가 몇개가 들어있는지 확인
            batch_size = images.size(0)

            # 모델 예측값 저장 (20개의 feature)
            preds_20 = model(images)

            # 20개의 예측값 중 우리가 비교할 11개의 offset, heading각, X 좌표값만 추출
            # preds_20[:, 0:2] : Offset, Heading
            # preds_20[:, 2::2] : x1, x2, ..., x9
            preds_11 = torch.cat([preds_20[:, 0:2], preds_20[:, 2::2]], dim=1)

            # 정답 데이터를 물리값으로 역스케일링
            # 배포용 모델에 저장해둔 mean과 std 버퍼를 가져와서 사용
            labels_physical = labels_normalized * model.std + model.mean

            # 오차값(절댓값)을 계산
            abs_error = torch.abs(preds_11 - labels_physical)
            # 매 오차를 누적하여 저장
            total_abs_error += torch.sum(abs_error, dim=0)
            # 계산된 총 이미지 개수를 누적하여 저장
            total_samples += batch_size

    # 평균 절대 오차(MAE) 계산
    # Numpy는 CPU RAM에 있는 데이터만 다룰 수 있음
    # tensor를 출력을 위해 Numpy로 변환하기 전에 데이터를 CPU로 옮겨오는 것
    feature_mae = (total_abs_error / total_samples).cpu().numpy()

    # feature 이름 저장
    feature_names = ["Offset", "Heading"] + [f"Waypoint X{i+1}" for i in range(9)]

    print("Feature별 MAE")

    # zip() : 2개의 list를 같은 순서끼리 짝지어주는 함수
    for name, mae in zip(feature_names, feature_mae):

        # heading의 경우 단위 deg, 나머지는 미터(m)
        unit = "" if name == "Heading" else "m"

        # :<12 12칸의 공간 확보 후 글씨 왼쪽 정렬
        print(f"{name:<12} : {mae:.4f} {unit}")
    print("="*30)

    # 전체 평균 MAE (11개 항목의 평균)
    print(f"Total Avg MAE : {np.mean(feature_mae):.4f}")

    # 11개의 각 feature별 MAE를 numpy 리스트 형태로 반환
    return feature_mae

# GPU 있으면 device를 GPU로, 없으면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 불러오기
model_path = "/content/drive/MyDrive/best_resnet_model.pt"

# MAE 계산 함수 호출
mae_results = calculate_feature_mae(model_path, test_loader, device)